In [1]:
import pandas as pd
import numpy as np
from rich import columns
from sympy.physics.units import length

# Exploration

In [2]:
airbnb = pd.read_csv("data/airbnb_train.csv")
airbnb.head()
print("coucou Antsa")

coucou Antsa


In [3]:
print("Colonnes du dataset :")
airbnb.columns

print(f"Taille du dataset : {len(airbnb)}")

Colonnes du dataset :
Taille du dataset : 22234


Type de données contenues dans le dataset

In [4]:
airbnb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22234 entries, 0 to 22233
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      22234 non-null  int64  
 1   log_price               22234 non-null  float64
 2   property_type           22234 non-null  object 
 3   room_type               22234 non-null  object 
 4   amenities               22234 non-null  object 
 5   accommodates            22234 non-null  int64  
 6   bathrooms               22183 non-null  float64
 7   bed_type                22234 non-null  object 
 8   cancellation_policy     22234 non-null  object 
 9   cleaning_fee            22234 non-null  bool   
 10  city                    22234 non-null  object 
 11  description             22234 non-null  object 
 12  first_review            17509 non-null  object 
 13  host_has_profile_pic    22178 non-null  object 
 14  host_identity_verified  22178 non-null

On voit que la majorité des données stockées sont de type 'object'.

Nombre de valeurs nulles dans le dataset (par colonne)

In [5]:
airbnb.isnull().sum()

id                           0
log_price                    0
property_type                0
room_type                    0
amenities                    0
accommodates                 0
bathrooms                   51
bed_type                     0
cancellation_policy          0
cleaning_fee                 0
city                         0
description                  0
first_review              4725
host_has_profile_pic        56
host_identity_verified      56
host_response_rate        5475
host_since                  56
instant_bookable             0
last_review               4716
latitude                     0
longitude                    0
name                         0
neighbourhood             2086
number_of_reviews            0
review_scores_rating      4978
zipcode                    303
bedrooms                    26
beds                        35
dtype: int64

Liste des colonnes dans lesquelles il y a des valeurs nulles

In [6]:
airbnb.columns[airbnb.isnull().any()]

Index(['bathrooms', 'first_review', 'host_has_profile_pic',
       'host_identity_verified', 'host_response_rate', 'host_since',
       'last_review', 'neighbourhood', 'review_scores_rating', 'zipcode',
       'bedrooms', 'beds'],
      dtype='object')

On remarque que dans les colonnes first_review, host_response_rate et last_review, les valeurs nulles sont importantes. On va donc regarder le pourcentage de valeurs nulles pour ces colonnes.

In [7]:
n = len(airbnb)

print(f"Pourcentage de valeurs nulles dans 'first_review' : {airbnb['first_review'].isnull().sum()/n}")
print(f"Pourcentage de valeurs nulles dans 'host_response_rate' : {airbnb['host_response_rate'].isnull().sum()/n}")
print(f"Pourcentage de valeurs nulles dans 'last_review' : {airbnb['last_review'].isnull().sum()/n}")

Pourcentage de valeurs nulles dans 'first_review' : 0.2125123684447243
Pourcentage de valeurs nulles dans 'host_response_rate' : 0.24624449042007737
Pourcentage de valeurs nulles dans 'last_review' : 0.21210758298102006


Le pourcentage de valeurs nulles étant assez important, on décide donc de drop les colonnes concernées.
On drop aussi 'description', car on sait d'ores et déjà qu'elle ne permettra pas d'améliorer les prédictions : aucune information utile sur l'appartement n'est donnée.

In [8]:
airbnb.drop(columns=["first_review", "host_response_rate", "last_review"], axis=1, inplace=True)
airbnb.drop("description", axis=1, inplace=True)
airbnb.columns

Index(['id', 'log_price', 'property_type', 'room_type', 'amenities',
       'accommodates', 'bathrooms', 'bed_type', 'cancellation_policy',
       'cleaning_fee', 'city', 'host_has_profile_pic',
       'host_identity_verified', 'host_since', 'instant_bookable', 'latitude',
       'longitude', 'name', 'neighbourhood', 'number_of_reviews',
       'review_scores_rating', 'zipcode', 'bedrooms', 'beds'],
      dtype='object')

Affichage de statistiques génériques (count, moyenne, écart-type, nombre d'élements uniques...)

In [9]:
airbnb.describe(include="all")

,id,log_price,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,...,instant_bookable,latitude,longitude,name,neighbourhood,number_of_reviews,review_scores_rating,zipcode,bedrooms,beds
count,2.223400e+04,22234.000000,22234,22234,22234,22234.000000,22183.000000,22234,22234,22234,...,22234,22234.000000,22234.000000,22234,20148,22234.000000,17256.000000,21931,22208.000000,22199.000000
unique,NaN,NaN,31,3,21160,NaN,NaN,5,5,2,...,2,NaN,NaN,22155,558,NaN,NaN,674,NaN,NaN
top,NaN,NaN,Apartment,Entire home/apt,{},NaN,NaN,Real Bed,strict,True,...,f,NaN,NaN,East Village Studio,Williamsburg,NaN,NaN,11211.0,NaN,NaN
freq,NaN,NaN,14635,12348,161,NaN,NaN,21622,9726,16401,...,16401,NaN,NaN,4,878,NaN,NaN,425,NaN,NaN
mean,1.122269e+07,4.783481,NaN,NaN,NaN,3.155573,1.236037,NaN,NaN,NaN,...,NaN,38.462971,-92.269305,NaN,NaN,20.670774,94.069077,NaN,1.264769,1.711473
std,6.080480e+06,0.718758,NaN,NaN,NaN,2.143870,0.586246,NaN,NaN,NaN,...,NaN,3.071679,21.670081,NaN,NaN,37.183731,7.782235,NaN,0.852819,1.254903
min,3.362000e+03,2.302585,NaN,NaN,NaN,1.000000,0.000000,NaN,NaN,NaN,...,NaN,33.339002,-122.510940,NaN,NaN,0.000000,20.000000,NaN,0.000000,0.000000
25%,6.202924e+06,4.317488,NaN,NaN,NaN,2.000000,1.000000,NaN,NaN,NaN,...,NaN,34.136082,-118.340633,NaN,NaN,1.000000,92.000000,NaN,1.000000,1.000000
50%,1.217425e+07,4.700480,NaN,NaN,NaN,2.000000,1.000000,NaN,NaN,NaN,...,NaN,40.662632,-76.994944,NaN,NaN,6.000000,96.000000,NaN,1.000000,1.000000
75%,1.639502e+07,5.220356,NaN,NaN,NaN,4.000000,1.000000,NaN,NaN,NaN,...,NaN,40.746358,-73.954599,NaN,NaN,23.000000,100.000000,NaN,1.000000,2.000000


On remarque que pour certaines colonnes (bed_type, host_identity_verified par ex.), il y a une valeur qui ressort souvent. Dans les parties qui vont suivre, on va essayer de voir la fréquence d'apparition de ces valeurs afin de les éliminer ou non.

Fréquence d'apparition des valeurs dites 'top' dans les colonnes éligibles (sans compter les valeurs nulles)

In [10]:
n = len(airbnb)

print(f"Pourcentage d'apparition de 'Real Bed' dans bed_type : {sum(airbnb[airbnb['bed_type'] == 'Real Bed'].count())/airbnb['bed_type'].count()}")
print(f"Pourcentage d'apparition de 'Entire home/apt' dans room_type : "
      f"{sum(airbnb[airbnb['room_type'] == 'Entire home/apt'].count())/airbnb['room_type'].count()}")
print(f"Pourcentage d'apparition de 'strict' dans bed_type : {sum(airbnb[airbnb['cancellation_policy'] == 'strict'].count())/airbnb['cancellation_policy'].count()}")
print(f"Pourcentage d'apparition de 'True' dans cleaning_fee : {sum(airbnb[airbnb['cleaning_fee'] == True].count())/airbnb['cleaning_fee'].count()}")
print(f"Pourcentage d'apparition de 't' dans host_has_profile_pic : "
      f"{sum(airbnb[airbnb['host_has_profile_pic'] == 't'].count())/airbnb['host_has_profile_pic'].count()}")
print(f"Pourcentage d'apparition de 't' dans host_identity_verified : "
      f"{sum(airbnb[airbnb['host_identity_verified'] == 't'].count())/airbnb['host_identity_verified'].count()}")
print(f"Pourcentage d'apparition de 'f' dans instant_bookable : {sum(airbnb[airbnb['instant_bookable'] == 'f'].count())/airbnb['instant_bookable'].count()}")
print(f"Pourcentage d'apparition de 'Williamsburg' dans neighbourhood : "
      f"{sum(airbnb[airbnb['neighbourhood'] == 'Williamsburg'].count())/airbnb['neighbourhood'].count()}")


Pourcentage d'apparition de 'Real Bed' dans bed_type : 23.004947377889717
Pourcentage d'apparition de 'Entire home/apt' dans room_type : 13.160295043626878
Pourcentage d'apparition de 'strict' dans bed_type : 10.390078258522983
Pourcentage d'apparition de 'True' dans cleaning_fee : 17.496986597103536
Pourcentage d'apparition de 't' dans host_has_profile_pic : 23.59040490576247
Pourcentage d'apparition de 't' dans host_identity_verified : 16.001668319956714
Pourcentage d'apparition de 'f' dans instant_bookable : 17.445893676351535
Pourcentage d'apparition de 'Williamsburg' dans neighbourhood : 1.0357851895969823


On remarque que certains pourcentages sont très importants

Vérification des colonnes dont le contenu ne peut pas être traité par un algorithme

In [11]:
rooms = airbnb["room_type"].unique()
beds_type = airbnb["bed_type"].unique()
cancellation = airbnb["cancellation_policy"].unique()
verified = airbnb["host_identity_verified"].unique()
online_booking = airbnb["instant_bookable"].unique()
cities = airbnb["city"].unique()
properties = airbnb["property_type"].unique()


print("Types de propriétés : ", properties)
print(f"Type de chambres : {rooms}")
print(f"Type de lits : {beds_type}")
print(f"Conditions d'annulation : {cancellation}")
print(f"Identité vérifiée : {verified}")
print(f"Réservation en ligne : {online_booking}")
print(f"Villes : {cities}")


Type de chambres : ['Private room' 'Entire home/apt' 'Shared room']
Type de lits : ['Real Bed' 'Pull-out Sofa' 'Futon' 'Airbed' 'Couch']
Conditions d'annulation : ['flexible' 'strict' 'moderate' 'super_strict_30' 'super_strict_60']
Identité vérifiée : ['f' 't' nan]
Réservation en ligne : ['t' 'f']
Villes : ['LA' 'NYC' 'DC' 'SF' 'Chicago' 'Boston']


On peut transformer ces colonnes en index pour simplifier le traitement de l'information par l'algorithme : c'est la prochaine étape de ce notebook.

# Entraînement

### Remplacer le type de propriété par un indice, cela permet à l’algo de l’utiliser

In [12]:
class CustomTransformation():

    def __init__(self):
        """
        Class simple pour convertir les type de propriétés en des indices numériques, utilisable pour un algo de machine learning
        """

        self.fitted = False # Indique si fit_transform a été utilisé, pour éviter d’utiliser transform sans que fit ait été appelé
        self.property2index = dict()# Dictionnaire qui va convertir le nom en indice
        self.rooms2index = dict()
        self.beds2index = dict()
        self.cancel2index = dict()
        self.online2index = dict()
        self.cities2index = dict()

        self.tf2index = {'t':True, 'f':False}

        self.max_index = 0 # Indique le dernier indice de la propriété.

    def fit_transform(self, dataset):

        self.fitted = True

        # Récupère les types de propriété (maison, appart etc...)


        self.property2index = {prop:i for (i, prop) in enumerate(properties)}
        self.max_index = max(list(self.property2index.values()))

        # transform
        return self.transform(dataset)
    
    def transform(self, dataset):
        # Transform les propriétés en indice
        dataset.loc[:, "property_type"] = dataset["property_type"].replace(self.property2index)


        # Ligne un peu moche qui fait en sorte de remplacer les lignes qui ont des noms de logement qui n’était pas dans l’entrainement
        dataset.loc[dataset["property_type"].map(type).eq(str), "property_type"] = np.nan

        
        # remplace les valeurs null
        dataset[dataset.bathrooms.isna()] = 0
        dataset[dataset.accommodates.isna()] = 0
        dataset[dataset.property_type.isna()] = self.max_index + 1
        return dataset

In [13]:
features_transformer = CustomTransformation()

airbnb.head()
airbnb_train = features_transformer.fit_transform(airbnb)
airbnb_train.head()

Tous les types de propriétés :  ['House' 'Apartment' 'Townhouse' 'Guest suite' 'Condominium' 'Timeshare'
 'Chalet' 'Guesthouse' 'Bungalow' 'Loft' 'In-law' 'Boat' 'Dorm' 'Other'
 'Bed & Breakfast' 'Camper/RV' 'Villa' 'Boutique hotel' 'Cabin' 'Hostel'
 'Hut' 'Yurt' 'Serviced apartment' 'Castle' 'Vacation home' 'Tent' 'Cave'
 'Tipi' 'Earth House' 'Island' 'Treehouse']


/var/folders/bb/zzg0xqxx14n56mdyj0ynd9xh0000gn/T/ipykernel_14697/2886256945.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset.loc[:, "property_type"] = dataset["property_type"].replace(self.property2index)
/var/folders/bb/zzg0xqxx14n56mdyj0ynd9xh0000gn/T/ipykernel_14697/2886256945.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  dataset[dataset.bathrooms.isna()] = 0


,id,log_price,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,...,instant_bookable,latitude,longitude,name,neighbourhood,number_of_reviews,review_scores_rating,zipcode,bedrooms,beds
0,5708593,4.317488,0,Private room,"{TV,""Wireless Internet"",Kitchen,""Free parking ...",3,1.0,Real Bed,flexible,False,...,t,33.782712,-118.134410,Island style Spa Studio,Long Beach,0,NaN,90804,0.0,2.0
1,14483613,4.007333,0,Private room,"{""Wireless Internet"",""Air conditioning"",Kitche...",4,2.0,Real Bed,strict,False,...,t,40.705468,-73.909439,"Beautiful and Simple Room W/2 Beds, 25 Mins to...",Ridgewood,38,86.0,11385,1.0,2.0
2,10412649,7.090077,1,Entire home/apt,"{TV,""Wireless Internet"",""Air conditioning"",Kit...",6,2.0,Real Bed,flexible,False,...,t,38.917537,-77.031651,2br/2ba luxury condo perfect for infant / toddler,U Street Corridor,0,NaN,20009,2.0,2.0
3,17954362,3.555348,0,Private room,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A...",1,1.0,Real Bed,flexible,True,...,f,40.736001,-73.924248,Manhattan view from Queens. Lovely single room .,Sunnyside,19,96.0,11104,1.0,1.0
4,9969781,5.480639,0,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.0,Real Bed,moderate,True,...,f,37.744896,-122.430665,Zen Captured Noe Valley House,Noe Valley,15,96.0,94131,2.0,2.0


In [14]:
class FeatureSelection():

    def __init__(self):
        """
        Class simple pour juste garder les colonnes qui nous intéresse
        N'a pas forcément l'air nécessaire, mais c'est pour être sur que j'applique bien le même process au train et au test
        """

    def fit_transform(self, dataset, y=None):
        return self.transform(dataset)
    
    def transform(self, dataset):
        new_dataset = dataset[["property_type", "accommodates", "bathrooms"]].copy()

        for col in new_dataset.columns:
            new_dataset[col] = pd.to_numeric(new_dataset[col]) # Converti tout en nombre
        return new_dataset
    
feature_selector = FeatureSelection()

airbnb_train = feature_selector.transform(airbnb)
airbnb_train.head()

,property_type,accommodates,bathrooms
0,0,3,1.0
1,0,4,2.0
2,1,6,2.0
3,0,1,1.0
4,0,4,1.0


### Apprentissage

In [15]:
from sklearn.model_selection import train_test_split

# Vous avez le droit d’utiliser les Pipeline et transform de sklearn : 
# https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
features_transformer = CustomTransformation()
feature_selector = FeatureSelection()

airbnb = pd.read_csv("data/airbnb_train.csv")

airbnb_train = features_transformer.fit_transform(airbnb)
airbnb_train = feature_selector.transform(airbnb_train)

X = airbnb_train.copy()
y = airbnb["log_price"]

# train cross validation
X_train, X_test, y_train, y_test = train_test_split(X, y)

Tous les types de propriétés :  ['House' 'Apartment' 'Townhouse' 'Guest suite' 'Condominium' 'Timeshare'
 'Chalet' 'Guesthouse' 'Bungalow' 'Loft' 'In-law' 'Boat' 'Dorm' 'Other'
 'Bed & Breakfast' 'Camper/RV' 'Villa' 'Boutique hotel' 'Cabin' 'Hostel'
 'Hut' 'Yurt' 'Serviced apartment' 'Castle' 'Vacation home' 'Tent' 'Cave'
 'Tipi' 'Earth House' 'Island' 'Treehouse']


/var/folders/bb/zzg0xqxx14n56mdyj0ynd9xh0000gn/T/ipykernel_14697/2886256945.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset.loc[:, "property_type"] = dataset["property_type"].replace(self.property2index)
/var/folders/bb/zzg0xqxx14n56mdyj0ynd9xh0000gn/T/ipykernel_14697/2886256945.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  dataset[dataset.bathrooms.isna()] = 0


Le score R2 est un score de regression, il vaut 1 si la prédiction est parfaite, 0 si la valeur prédite est la moyenne de $y$. Et des scores négatifs si la prédiction est moins bonne que prédire la moyenne (donc vraiment mauvais)

In [16]:
from sklearn.svm import LinearSVR
from sklearn.metrics import r2_score

model = LinearSVR()

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)


print(f"Score en entrainenement : {r2_score(y_true=y_train, y_pred=y_pred_train)}")
print(f"Score en cross validation : {r2_score(y_true=y_test, y_pred=y_pred_test)}")


Score en entrainenement : 0.32065398368228504
Score en cross validation : 0.3149556533763408


/Users/amyrazafi/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## Prédiction sur le fichier de test

In [17]:
airbnb_test = pd.read_csv("airbnb_test.csv")

# J’applique le même traitement que mon fichier entraînement
final_X_test = features_transformer.transform(airbnb_test)
final_X_test = feature_selector.transform(final_X_test)

final_X_test.tail()

FileNotFoundError: [Errno 2] No such file or directory: 'airbnb_test.csv'

In [ ]:
y_final_prediction = model.predict(final_X_test)
print(y_final_prediction)

## Sauvegarde dans le fichier de prédiction

In [ ]:
prediction_example = pd.read_csv("prediction_example.csv")
prediction_example["logpred"] = y_final_prediction

prediction_example.to_csv("MaPredictionFinale.csv", index=False) # index=False pour éviter d’ajouter l’index interne à pandas
# Voilà !

## Test de votre fichier

In [ ]:
def estConforme(monFichier_csv):
    votre_prediction = pd.read_csv(monFichier_csv)

    fichier_exemple = pd.read_csv("prediction_example.csv")

    assert votre_prediction.columns[1] == fichier_exemple.columns[1], f"Attention, votre colonne de prédiction doit s'appeler {fichier_exemple.columns[1]}, elle s'appelle '{votre_prediction.columns[1]}'"
    assert len(votre_prediction) == len(fichier_exemple), f"Attention, vous devriez avoir {len(fichier_exemple)} prédiction dans votre fichier, il en contient '{len(votre_prediction)}'"

    assert np.all(votre_prediction.iloc[:,0] == fichier_exemple.iloc[:, 0])

    print("Fichier conforme!")

estConforme("MaPredictionFinale.csv")

# Ce que je vais faire de mon côté

In [ ]:
# Vous n’avez pas accès à ce fichier, c’est normal, ce sont les vrais prédictions
# ===============================================================================
# true_test = pd.read_csv("../true_prediction.csv") 
# ==========================================================

# votre_prediction = pd.read_csv("MaPredictionFinale.csv")["logpred"]
# print(r2_score(y_pred=votre_prediction, y_true=true_test["log_price"]))

# votre_prediction = pd.read_csv("prediction_example.csv")["logpred"] # Devrait avoir ~0
# print(r2_score(y_pred=votre_prediction, y_true=true_test["log_price"]))